# LLM Agents: Wikipedia Navigation Game

This notebook demonstrates building LLM agents using the Inspect AI framework. The agents navigate between Wikipedia pages by following links, simulating the "Wikipedia Game." We progressively build more sophisticated agents:

1. **Simple Arithmetic Agent** -- A basic agent with tool-calling to solve arithmetic tasks
2. **WikiGame Agent** -- An agent that navigates Wikipedia pages using content and link tools
3. **Elicitation Improvements** -- Prompt engineering, ReAct reasoning, chat history management, and reflexion tools

## Setup

In [1]:
import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any, Literal, Optional

import inspect_ai
from inspect_ai.agent import Agent, AgentState, agent
import inspect_ai.model as inspect_ai_model
from inspect_ai.model import (
    ChatMessageAssistant, ChatMessageUser, ChatMessageSystem,
    ChatMessageTool, get_model, execute_tools,
)
from inspect_ai.scorer import match
from inspect_ai.dataset import Sample, json_dataset, hf_dataset
from inspect_ai.tool import tool, Tool, ToolCall, tool_with
from inspect_ai.agent import run
from inspect_ai import Task, task, eval
from inspect_ai.agent import as_solver

import wikipedia
from dotenv import load_dotenv
from openai import OpenAI
from wikipedia import DisambiguationError, PageError, WikipediaPage

from utils import countrylist, evaluate_expression, wiki_pairs, execute_tools, extract_answer

load_dotenv()

EVAL_MODEL = "openai/gpt-4o-mini"
os.environ["INSPECT_EVAL_MODEL"] = EVAL_MODEL

## Simple Arithmetic Agent

A minimal agent with tool-calling to solve arithmetic expressions.

In [2]:
class ArithmeticTask:

    def __init__(self, num1: int | float, num2: int | float, operations: Optional[list[str]] = None):
        self.num1 = num1
        self.num2 = num2
        self.operations = operations if operations else ["+", "-", "*", "/", "**", "//", "%"]
        self.current_task_number = 0

    def _generate_answers(self) -> list[str]:
        """
        Generates a list of the correct answers for all the possible tasks

        Returns:
            list[str]: A list of the correct answers for all the possible tasks
        """
        ans = []
        for op in self.operations:
            try:
                ans.append(str(evaluate_expression(f"{self.num1} {op} {self.num2}")))
            except Exception as e:
                ans.append(f"Exception: {str(e)}")
        return ans
    
    @property
    def get_current_task(self) -> str:
        return f"{self.num1} {self.operations[self.current_task_number]} {self.num2}"
    def update_current_task(self) -> None:
        """
        Increments self.current_task_number by one (modulo the number of operations)
        """
        self.current_task_number = (self.current_task_number + 1) % len(self.operations)

    def get_current_instruction(self) -> ChatMessageUser:
        return ChatMessageUser(content=f"Calculate the following expression: {self.get_current_task}, output your answer in the format <ANSWER>NUMBER</ANSWER> where NUMBER is a float.")

In [3]:
@tool
def calculate(): 
    async def execute(expression : str) -> str:
        """
        This tool evaluates a given arithmetic expression.
        For example, the expression can be "5 + 8", and the tool will output "13".

        Args:
            expression: the expression you wish to evaluate.
        Returns:
            The evaluated result, or an exception if it occurred during calculation.
        """
        try:
            return str(evaluate_expression(expression))
        except Exception as e:
            return f"Exception: {str(e)}"
    return execute

In [4]:
@agent
def arithmetic_agent(task : ArithmeticTask):
    async def execute(state: AgentState) -> AgentState:
        success = False
        ans = ["Wrong"] * len(task.operations)
        correct_ans = task._generate_answers()
        while not success:
            state.messages.append(task.get_current_instruction())
            state.output = await get_model().generate(
                input=state.messages,                          
                tools=[calculate()],
            )
            state.messages.append(state.output.message)

            if state.output.message.tool_calls:                      
                messages, state.output = await execute_tools(
                    state.messages, [calculate()]
                )
                state.messages.extend(messages)
                # pass answer back to llm
                state.output = await get_model().generate(
                    input=state.messages,                          
                )
                state.messages.append(state.output.message)
            
            try:
                llm_answer = extract_answer(state.output.message.content)
                if llm_answer == correct_ans[task.current_task_number]:
                    ans[task.current_task_number] = llm_answer
                    task.update_current_task()
                else:
                    state.messages.append(ChatMessageUser(content="Incorrect answer. Try again."))
            except IndexError:
                state.messages.append(ChatMessageUser(content="Error: cannot extract answer"))
            if all(ans_i == correct_ans[i] for i, ans_i in enumerate(ans)):
                success = True

        return state
    return execute

In [5]:
@task
def agent_task() -> str:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=40)

# eval(agent_task(), solver = as_solver(arithmetic_agent(task = ArithmeticTask(3, 5))))

## WikiGame Agent

The Wikipedia Game: navigate between pages using only links found in page content.

### Utility Functions

In [6]:
def get_page(title: str) -> WikipediaPage:
    """
    Get a Wikipedia page object given a title. If the title is ambiguous, choose the first option.
    If the title is not found, try to find a similar title.

    Args:
        title (str): The title of the Wikipedia page

    Returns:
        WikipediaPage: The Wikipedia page
    """
    try:
        return wikipedia.page(title, auto_suggest=False, redirect=True)
    except DisambiguationError as e:
        return wikipedia.page(e.options[0], auto_suggest=False, redirect=True)
    except PageError:
        return wikipedia.page(title, auto_suggest=True, redirect=True)

In [7]:
def get_permitted_links(current_page: WikipediaPage) -> list[str]:
    """
    Get "permitted" links (i.e. links that are in the content of the page) from a Wikipedia page.

    Args:
        current_page (WikipediaPage): The current Wikipedia page

    Returns:
        list[str]: A list of permitted links from current_page

    """
    permitted = []
    for link in current_page.links:
        if link.lower() in current_page.content.lower() and link != current_page.title:
            permitted.append(link)
    return permitted




### WikiGame Class

In [8]:
class WikiGame:
    def __init__(
        self,
        starting_page: str,
        goal_page: str,
    ):
        """
        This task simulates the Wikipedia game, where the agent starts on one Wikipedia page and
        attempts to navigate to a goal page using only links found in the main content of Wikipedia
        pages.

        Args:
            starting_page (str): The page the agent starts on.
            goal_page (str): The page the agent is trying to reach.

        Attributes:
            page_history (list[str]): The history of pages visited by the agent.
            starting_page (WikipediaPage): The starting page of the game.
            goal_page (WikipediaPage): The goal page of the game.
            current_page (WikipediaPage): The current page the agent is on.

        """
        self.page_history: list[str] = [starting_page]
        self.starting_page: WikipediaPage = self.get_page(starting_page)
        self.goal_page: WikipediaPage = self.get_page(goal_page)
        self.current_page: WikipediaPage = self.starting_page

    # ========================= Helper Functions (given) =========================

    # Get page and page summary
    @staticmethod
    def get_page(title: str) -> WikipediaPage:
        """
        Get a Wikipedia page object given a title. If the title is ambiguous, choose the first
        option. If the title is not found, try to find a similar title.

        Args:
            title (str): The title of the Wikipedia page

        Returns:
            WikipediaPage: The Wikipedia page
        """
        try:
            return wikipedia.page(title, auto_suggest=False, redirect=True)
        except DisambiguationError as e:
            return wikipedia.page(e.options[0], auto_suggest=False, redirect=True)
        except PageError:
            return wikipedia.page(title, auto_suggest=True, redirect=True)

    def get_page_summary(self, page: WikipediaPage | None = None) -> str:
        """
        Get summary of a wikipedia page, to the last full stop within the first 500 characters.
        This can be used to give a brief overview of a page to the agent.

        Args:
            page (WikipediaPage): The Wikipedia page object.

        Returns:
            str: The summary of the Wikipedia page.
        """
        page = page if page else self.goal_page
        summary = page.content[:500]
        last_period_index = summary.rfind(".")
        return summary[: last_period_index + 1] if last_period_index != -1 else summary

    # Get and check permitted links
    def get_permitted_links(self) -> list[str]:
        """
        Returns a list of permitted links (i.e. links in the main page content) for the current page.

        Returns:
            list[str]: The permitted links.
        """
        all_links = self.current_page.links
        content_lower = self.current_page.content.lower()
        permitted_links = [link for link in all_links if link.lower() in content_lower]
        if self.current_page.title in permitted_links:
            permitted_links.remove(self.current_page.title)
        return permitted_links

    def is_permitted_link(self, link: str) -> bool:
        """
        Returns True if the link is in the permitted links for the current page, False otherwise.

        Args:
            link (str): The link to check.

        Returns:
            bool: True if the link is permitted, False otherwise
        """
        return link.lower() in (x.lower() for x in self.get_permitted_links())

    # ========================= Task State Management (given) =========================

    def check_win(self) -> bool:
        return self.current_page == self.goal_page

### WikiGame Tools

In [9]:
@tool
def GetContentTool(game : WikiGame) -> Tool:
    async def execute() -> str:
        """
        Get all the content for the wikipedia page you are currently on. Anything which corresponds to a link is wrapped in <link></link> tags.

        Args:
            None

        Returns:
            str: The content of the page with any accessible links wrapped in <link></link> tags
        """
        content = game.current_page.content
        permitted_links = game.get_permitted_links()
        for word in sorted(permitted_links, key=len, reverse=True):
            content = re.sub(
                r"""(\s|[,.)!?;:'"])(""" + re.escape(word) + r""")(\s|[,.)!?;:'"s])""",
                r"\1<link>\2</link>\3",
                content,
                count=1,
                flags=re.IGNORECASE,
            )
        return content

    return execute
     
@tool 
def MovePageTool(game : WikiGame) -> Tool:
    async def execute(page: str) -> str:
        """
        Move to a new wikipedia page by clicking on a link in the current page content. Modifies the game state in place.

        Args:
            page: The title of the page you want to move to. This must be accessible from the current page (and be a different page), or the move will fail.

        Returns:
            str: A message indicating whether the move was successful
        """ 
        page_no_underscore = page.replace("_", " ")
        try:
            if game.is_permitted_link(page):
                new_page = game.get_page(page)
            elif game.is_permitted_link(page_no_underscore):
                new_page = game.get_page(page_no_underscore)
            else:
                return "Error: Link is not on page!"
            game.current_page = new_page
            return f"Success: Move to {game.current_page.title}"

        except Exception as e:
            return f"Error: {str(e)}"

    return execute

### WikiAgent

In [10]:
@agent 
def WikiAgent(tools : list[Tool], game: WikiGame):
    system_instruction = ChatMessageSystem(content="You are an agent that plays the Wikipedia game. You will be given a start page and a goal page, and you will need to navigate to goal page from start page using only links in each page. You want to minize the number of intermediate links.")
    on_page_instruction = ChatMessageUser(content=f"You are on this page: {game.current_page.title}. You need to reach {game.goal_page.title}.")
    next_step_instruction = ChatMessageUser(content="You should make the next action. Make only 1 tool call for each message.")

    async def instruction_refresh() -> None:
        nonlocal system_instruction, on_page_instruction, next_step_instruction
        on_page_instruction = ChatMessageUser(content=f"You are on this page: {game.current_page.title}. You need to reach: {game.goal_page.title}.")
        
    async def _reset_history(state : AgentState):
        state.messages = []
        state = await _start(state)
        return state
    
    async def _start(state: AgentState) -> AgentState:
        state.messages.extend([system_instruction, on_page_instruction])
        return state
    
    async def _handle_tool_calls(state: AgentState) -> AgentState:
        messages, state.output = await execute_tools(
            state.messages, tools
        )
        state.messages.extend(messages)
        if state.output.message.tool_calls[0].function == "MovePageTool" and "success" in messages[-1].content.lower():
            await instruction_refresh()
            state = await _reset_history(state)
        return state
    
    async def execute(state : AgentState) -> AgentState:
        i = 0
        max_iter = 20
        success = False
        await _start(state)
        while not success:
            state.messages.append(next_step_instruction)
            state.output = await get_model().generate(
                input=state.messages,                          
                tools=tools
            )
            state.messages.append(state.output.message)

            if state.output.message.tool_calls:                      
                state = await _handle_tool_calls(state)

            if game.check_win():
                success = True
            
            i += 1
            if i >= max_iter:
                raise RuntimeError("Max iteration exceeded!")

        return state
    
    return execute

### Running the WikiAgent

In [ ]:
game_1 = WikiGame("Elizabeth I", "United States")
tool_list = [GetContentTool(game_1), MovePageTool(game_1)]
@task
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=80)
eval(solver = as_solver(WikiAgent(tools = tool_list, game = game_1)), tasks = wiki_task(),)

In [ ]:
game_2 = WikiGame("County Seat", "Saint Pierre and Miquelon")
tool_list = [GetContentTool(game_2), MovePageTool(game_2)]
@task
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=80)
eval(solver = as_solver(WikiAgent(tools = tool_list, game = game_2)), tasks = wiki_task(),)

Output()

## Elicitation Techniques

Improving agent performance through prompt engineering, ReAct reasoning, chat history management, and reflexion.

In [ ]:
os.environ["INSPECT_EVAL_MODEL"] = "openai/gpt-4o-mini"

### Improved Prompting

In [ ]:
@agent
def WikiAgentPrompting(tools: list[Tool], game: WikiGame) -> Agent:
    system_instruction = ChatMessageSystem(content="You are an agent that plays the Wikipedia game. You will be given a start page and a goal page, and you will need to navigate to goal page from start page using only links in each page. You want to minize the number of intermediate links.")
    on_page_instruction = ChatMessageUser(content=f"You are on this page: {game.current_page.title}. You need to reach {game.goal_page.title}.")
    next_step_instruction = ChatMessageUser(content="You should make the next action. Make only 1 tool call for each message.")

    async def instruction_refresh() -> None:
        nonlocal system_instruction, on_page_instruction, next_step_instruction
        on_page_instruction = ChatMessageUser(content=f"You are on this page: {game.current_page.title}. You need to reach: {game.goal_page.title}.")
        
    async def _reset_history(state : AgentState):
        state.messages = []
        state = await _start(state)
        return state
    
    async def _start(state: AgentState) -> AgentState:
        state.messages.extend([system_instruction, on_page_instruction])
        return state
    
    async def _handle_tool_calls(state: AgentState) -> AgentState:
        messages, state.output = await execute_tools(
            state.messages, tools
        )
        state.messages.extend(messages)
        if state.output.message.tool_calls[0].function == "MovePageTool" and "success" in messages[-1].content.lower():
            await instruction_refresh()
            state = await _reset_history(state)
        return state
    
    async def execute(state : AgentState) -> AgentState:
        i = 0
        max_iter = 30
        success = False
        await _start(state)
        while not success:
            state.messages.append(next_step_instruction)
            state.output = await get_model().generate(
                input=state.messages,                          
                tools=tools
            )
            state.messages.append(state.output.message)

            if state.output.message.tool_calls:                      
                state = await _handle_tool_calls(state)

            if game.check_win():
                success = True
            
            i += 1
            if i >= max_iter:
                raise RuntimeError("Max iteration exceeded!")

        return state
    
    return execute

Compare baseline `WikiAgent` vs `WikiAgentPrompting` on a harder path:

In [ ]:
game = WikiGame("Mandate of Heaven", "Doric Greek")
tool_list = [GetContentTool(game), MovePageTool(game)]
@task
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=80)
eval(solver = as_solver(WikiAgent(tools = tool_list, game = game)), tasks = wiki_task(),)

In [ ]:
game = WikiGame("Mandate of Heaven", "Doric Greek")
tool_list = [GetContentTool(game), MovePageTool(game)]
@task
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=80)
eval(solver = as_solver(WikiAgentPrompting(tools = tool_list, game = game)), tasks = wiki_task(),)

### ReAct Framework

The ReAct pattern separates reasoning from action: the agent first reasons about its next step, then takes an action.

In [ ]:
@agent
def WikiAgentReAct(tools: list[Tool], game: WikiGame) -> Agent:
    system_instruction = ChatMessageSystem(content = f"You are a wikipedia-racing AI. Your goal is to reach {game.goal_page.title} by accessing links from wikipedia pages. Your current page is {game.current_page.title}.")
    on_page_instruction = ChatMessageUser(content = f"""You are currently on page: {game.current_page.title}. Make sure you start by reasoning about what steps you should take to get to the article on {game.goal_page.title}. When coming up with a strategy, make sure to pay attention to the path you have already taken, and if your current strategy doesn't seem to be working out, try something else. In case you're unsure, {game.goal_page.title} has the following summary:\n\n[Begin Summary]\n{game.get_page_summary(game.goal_page)}\n[End Summary]\n\nThe path you have taken so far is {" -> ".join(game.page_history)}.
            """)
    next_step_reasoning = ChatMessageUser(content="You should reason about what to do for the next step.")
    next_step_action = ChatMessageUser(content="You should make the next action. Make only 1 tool call for each message.")

    async def _reset_history(state : AgentState):
        state.messages = []
        state = await _start(state)
        return state
    
    async def instruction_refresh() -> None:
        nonlocal system_instruction, on_page_instruction
        on_page_instruction = ChatMessageUser(content=f"You are on this page: {game.current_page.title}. You need to reach: {game.goal_page.title}.")

    async def generate_reason(state : AgentState) -> AgentState:
        state.messages.append(next_step_reasoning)
        state.output = await get_model().generate(
            input=state.messages,                          
            tools=tools,
            tool_choice="none"
        )
        state.messages.append(state.output.message)
        return state
    
    async def generate_action(state : AgentState) -> AgentState:
        state.messages.append(next_step_action)
        state.output = await get_model().generate(
            input=state.messages,                          
            tools=tools
        )
        state.messages.append(state.output.message)
        return state
        
    async def _start(state: AgentState) -> AgentState:
        state.messages.extend([system_instruction, on_page_instruction])
        return state
    
    async def _handle_tool_calls(state: AgentState) -> AgentState:
        messages, state.output = await execute_tools(
            state.messages, tools
        )
        state.messages.extend(messages)
        if state.output.message.tool_calls[0].function == "MovePageTool" and "success" in messages[-1].content.lower():
            await instruction_refresh()
            state = await _reset_history(state)
        return state    
    
    async def execute(state : AgentState) -> AgentState:
        i = 0
        max_iter = 30
        success = False
        await _start(state)
        while not success:
            # reasoning
            state = await generate_reason(state)
            # action
            state = await generate_action(state)

            if state.output.message.tool_calls:                      
                state = await _handle_tool_calls(state)

            if game.check_win():
                success = True
            
            i += 1
            if i >= max_iter:
                raise RuntimeError("Max iteration exceeded!")

        return state
    
    return execute

Compare `WikiAgentPrompting` vs `WikiAgentReAct`:

In [ ]:
# Run the game with WikiAgentPrompting
game = WikiGame("Balto-Slavic languages", "Netscape Navigator 9")
tool_list = [GetContentTool(game), MovePageTool(game)]
@task
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=80)

eval(solver = as_solver(WikiAgentPrompting(tools = tool_list, game = game)), tasks = wiki_task(),)

In [ ]:
# Run the game with WikiAgentReAct
game = WikiGame("Balto-Slavic languages", "Netscape Navigator 9")
tool_list = [GetContentTool(game), MovePageTool(game)]
@task
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=80)

eval(solver = as_solver(WikiAgentReAct(tools = tool_list, game = game)), tasks = wiki_task(),)

### Chat History Management

Preserve the full conversation history but replace verbose page content with summaries to manage context length.

In [ ]:
@agent 
def WikiAgentHistory(tools : list[Tool], game: WikiGame, verbose : bool = True):
    system_instruction = ChatMessageSystem(content = f"You are a wikipedia-racing AI. Your goal is to reach {game.goal_page.title} by accessing links from wikipedia pages. Your current page is {game.current_page.title}.")
    on_page_instruction = ChatMessageUser(content = f"""You are currently on page: {game.current_page.title}. Make sure you start by reasoning about what steps you should take to get to the article on {game.goal_page.title}. When coming up with a strategy, make sure to pay attention to the path you have already taken, and if your current strategy doesn't seem to be working out, try something else. In case you're unsure, {game.goal_page.title} has the following summary:\n\n[Begin Summary]\n{game.get_page_summary(game.goal_page)}\n[End Summary]\n\nThe path you have taken so far is {" -> ".join(game.page_history)}.
            """)
    next_step_reasoning = ChatMessageUser(content="You should reason about what to do for the next step.")
    next_step_action = ChatMessageUser(content="You should make the next action. Make only 1 tool call for each message.")

    async def _reset_history(state : AgentState, previous_page : str):
        for message in state.messages:
            if isinstance(message, ChatMessageTool) and message.function == "GetContentTool" and "Wikipedia page content for page" not in message.content:
                message.content = f"Wikipedia page content for page {previous_page} was output here, but has been removed for brevity."
        return state
    
    async def instruction_refresh() -> None:
        nonlocal system_instruction, on_page_instruction
        on_page_instruction = ChatMessageUser(content=f"You are on this page: {game.current_page.title}. You need to reach: {game.goal_page.title}.")

    async def generate_reason(state : AgentState) -> AgentState:
        state.messages.append(next_step_reasoning)
        state.output = await get_model().generate(
            input=state.messages,                          
            tools=tools,
            tool_choice="none"
        )
        state.messages.append(state.output.message)
        return state
    
    async def generate_action(state : AgentState) -> AgentState:
        state.messages.append(next_step_action)
        state.output = await get_model().generate(
            input=state.messages,                          
            tools=tools
        )
        state.messages.append(state.output.message)
        return state
        
    async def _start(state: AgentState) -> AgentState:
        state.messages.extend([system_instruction, on_page_instruction])
        return state
    
    async def _handle_tool_calls(state: AgentState) -> AgentState:
        messages, state.output = await execute_tools(
            state.messages, tools
        )
        state.messages.extend(messages)
        if state.output.message.tool_calls[0].function == "MovePageTool" and "success" in messages[-1].content.lower():
            previous_page = game.current_page.title
            await instruction_refresh()
            state = await _reset_history(state, previous_page)
        return state
    
    async def execute(state : AgentState) -> AgentState:
        i = 0
        max_iter = 30
        success = False
        await _start(state)
        while not success:
            # reasoning
            state = await generate_reason(state)
            # action
            state = await generate_action(state)

            if state.output.message.tool_calls:                      
                state = await _handle_tool_calls(state)

            if game.check_win():
                success = True
            
            i += 1
            if i >= max_iter:
                raise RuntimeError("Max iteration exceeded!")

        return state
    
    return execute

Compare `WikiAgentReAct` vs `WikiAgentHistory`:

In [ ]:
game = WikiGame("Blavatnik School of Government", "Free Thai Movement")
tool_list = [GetContentTool(game), MovePageTool(game)]

@task 
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=120)

eval(solver = as_solver(WikiAgentReAct(tools = tool_list, game = game)), tasks = wiki_task(),)

In [ ]:
game = WikiGame("Blavatnik School of Government", "Free Thai Movement")
tool_list = [GetContentTool(game), MovePageTool(game)]

@task 
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=120)

eval(solver = as_solver(WikiAgentHistory(tools = tool_list, game = game)), tasks = wiki_task(),)

### Reflexion: Path Testing Tool

A tool that lets the agent test whether a proposed sequence of links actually connects.

In [ ]:
@tool
def TestPathTool(game : WikiGame) -> Tool:
    async def execute(path: str) -> str:
        """
        Test a path of wikipedia pages to see if it leads to the goal page. The path should be a series of page titles separated by '->'. 

        Args:
            path (str): The path to test formatted as a series of wikipedia page titles separated by '->'. The path must start with the current page title. The path doesn't have to end with the goal page title.

        Returns:
            str: The result of the test. Success if the path leads to the goal page. Otherwise returns failure, and where the path failed.
        """
        try:
            path = [node.strip() for node in path.split("->")]
            if not path:
                return "Error: No path found after parsing"
            current_node = path[0]
            path_nodes = path[1:]
            if not path_nodes:
                return "Error: Only a single node in path."
            for node in path_nodes:
                permitted_links = get_permitted_links(get_page(current_node))
                if node not in permitted_links:
                    return f"Error: {node} not in {current_node} page"
            return "Success: the given path works!"
        except Exception as e:
            return f"Error: {str(e)}"
            
    return execute

In [ ]:
game = WikiGame("Blavatnik School of Government", "Free Thai Movement")
tool_list = [GetContentTool(game), MovePageTool(game), TestPathTool(game)]
@task   
def wiki_task() -> Task:
    return Task(dataset = [Sample(input = "", target = "")], message_limit=120)

eval(solver = as_solver(WikiAgentHistory(tools = tool_list, game = game)), tasks = wiki_task())